# 📗 Docker로 서빙 API 이미지 만들기: 노무 상담 API를 이미지로 빌드해 GHCR에 올리기

<div class="day78-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">실행 위치:</strong> 내 PC 노트북(VM의 API 호출, GitHub 실행 기록 읽기), 내 PC 터미널(git commit, push), GCP VM 터미널(docker build, run, pull), GitHub 화면(Actions, Packages). <strong style="font-size:1em;color:#1f3a5f;">이 교안은 RunPod를 쓰지 않습니다.</strong><br><strong style="font-size:1em;color:#1f3a5f;">준비:</strong> 교안 00을 마쳐 GitHub에 공개 저장소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code>가 있고, VM에 같은 저장소를 clone했고, 실습자료/.env에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_OWNER</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_READ_TOKEN</code>이 적혀 있어야 합니다.<br><strong style="font-size:1em;color:#1f3a5f;">비용:</strong> GCP VM(무료 체험 크레딧)만 씁니다. 공개 저장소의 Actions와 공개 패키지는 무료입니다.</div>

#### **서빙 API 이미지**

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">베이스 이미지 위에 잠금 파일(설치할 라이브러리 버전을 적어 둔 파일)대로 설치한 라이브러리와 서버 코드를 레이어로 쌓은 이미지입니다. vLLM 주소와 키 같은 설정은 컨테이너를 실행할 때 환경 변수로 넣습니다.</strong></p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 같은 이미지를 어디서나 그대로 실행하려면 장소마다 다른 값은 밖에서 넣어야 합니다. 키를 이미지에 넣으면 이미지를 받은 누구나 꺼낼 수 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 코드를 고칠 때마다 이미지를 레지스트리에 올리고, 서버가 태그로 골라 받아 실행하는 컨테이너 배포입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/10_serving_image_concept.png" width="1050" alt="서빙 API 이미지의 구성: 베이스 이미지, 잠금 파일로 설치한 라이브러리, 서버 코드 층과, 실행 때 컨테이너에 환경 변수로 넣는 vLLM 주소와 키">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무, 전월세 상담 API(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.managed_main</code>)를 이미지로 만듭니다. VM에서 빌드해 환경 변수로 설정을 넣어 실행하고, GitHub Actions가 같은 이미지를 빌드해 GHCR(GitHub의 컨테이너 레지스트리)에 올리게 합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/11_lesson01_goals.png" width="1050" alt="학습 목표 그림: 1 서빙 API Dockerfile 읽기, 2 VM에서 빌드하고 환경 변수로 실행, 3 태그를 붙여 GHCR에 올리기, 4 GHCR 이미지를 VM에서 받아 실행">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day78-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span> (그림의 번호와 같습니다)

1. 서빙 API의 Dockerfile을 읽고 잠금 파일, 코드, 실행 명령을 어떤 순서로 두는지 설명합니다.
2. VM에서 서빙 API 이미지를 빌드하고, 설정을 환경 변수로 넣어 컨테이너로 실행합니다.
3. 이미지 이름과 태그를 정하고, GitHub Actions로 이미지를 빌드해 GHCR에 올립니다.
4. 레지스트리의 이미지를 VM에서 받아 실행합니다.

</div>

## 1. 실습 준비

### 1-1. VM의 저장소와 포트 확인

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM 터미널에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code> 저장소를 최신 상태로 받고, 8080 포트를 쓰는 컨테이너가 남아 있지 않은지 확인합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
cd ~/serving-deploy
pwd   # 끝이 serving-deploy이면 맞습니다
git pull
ls
docker ps
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd 폴더</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pwd</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">그 폴더로 이동하고, 지금 폴더의 경로를 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git pull</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">GitHub 저장소의 새 커밋을 받아 VM의 저장소에 합칩니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ls</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 폴더의 파일과 폴더 목록을 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker ps</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">실행 중인 컨테이너 목록을 보여 줍니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker ps</code> 목록에 컨테이너가 남아 있으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker rm -f &lt;컨테이너 이름&gt;</code>으로 지웁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;컨테이너 이름&gt;</code>은 맨 오른쪽 NAMES 열의 값입니다.</p>
</div>

### 1-2. 내 PC 노트북 준비

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">공용 준비 코드에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 VM 주소와 GitHub 아이디를 더했습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 설정값 ----
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

day_dir = Path(".").resolve()       # 일차 폴더(day78_모델배포_Docker_CICD)

# 키와 주소는 실습자료/.env 에 모아 둡니다(화면에 출력하지 않습니다)
env_path = day_dir.parent / ".env"
load_dotenv(env_path, override=True)

# ---- VM 주소, GitHub 아이디 ----
# VM 의 서빙 API 주소(방화벽 규칙으로 8080 포트를 열었습니다)
VM_URL = "http://" + os.environ["VM_HOST"] + ":8080"

# GitHub 아이디(serving-deploy 저장소의 주인)
GITHUB_OWNER = os.environ["GITHUB_OWNER"]

# 노트북이 응답을 기다리는 최대 시간(초)
REQUEST_TIMEOUT = 30

## 2. 서빙 API Dockerfile 읽기

### 2-1. 명령의 순서

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">라이브러리를 설치하는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RUN</code>과 환경 변수를 정하는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ENV</code>가 더해졌습니다. 드물게 바뀌는 것을 위에, 자주 바뀌는 서버 코드를 아래에 둡니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving-deploy/Dockerfile</strong></div>

```dockerfile
# 노무, 전월세 상담 서빙 API 이미지(교안 01)

# (1) 베이스 이미지: 파이썬 3.12 의 slim 버전(서버 실행에 필요한 것만 든 작은 버전). 태그를 꼭 적는다
FROM python:3.12-slim

# (2) uv 실행 파일만 공식 uv 이미지에서 복사한다. 버전을 적어 두면 언제 빌드해도 같은 uv 를 쓴다
COPY --from=ghcr.io/astral-sh/uv:0.13.7 /uv /uvx /bin/

# (3) 이후 명령이 실행될 작업 폴더
WORKDIR /app

# (4) 라이브러리 목록과 잠금 파일만 먼저 복사해 설치한다. 코드만 고치면 이 두 층은 캐시를 그대로 쓴다
#     --locked: uv.lock 에 적힌 버전 그대로, --no-dev: 테스트 도구 제외, --no-cache: 내려받은 파일을 이미지에 남기지 않음
COPY pyproject.toml uv.lock ./
RUN uv sync --locked --no-dev --no-cache

# (5) 자주 바뀌는 서버 코드는 마지막에 복사한다
COPY serving_app ./serving_app

# (6) 설치한 가상환경의 실행 파일을 먼저 찾게 하고, 키 DB 위치를 정한다(배포 때 이 폴더에 볼륨을 붙인다)
ENV PATH="/app/.venv/bin:$PATH" \
    SERVICE_DB_PATH=/app/data/service.db

# (7) 이 이미지가 쓰는 포트를 적어 둔다(표시일 뿐이고, 실제로 여는 것은 docker run -p 와 compose 의 ports)
EXPOSE 8080

# (8) 컨테이너를 켜면 실행할 명령(exec 형식). 컨테이너 밖에서 들어오려면 0.0.0.0 에 묶는다
CMD ["uvicorn", "serving_app.managed_main:app", "--host", "0.0.0.0", "--port", "8080"]
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">FROM python:3.12-slim</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">파이썬 3.12가 든 작은 데비안 이미지를 베이스 이미지로 씁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY --from=ghcr.io/astral-sh/uv:0.13.7 /uv /uvx /bin/</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">공식 uv 이미지 안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uvx</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/bin/</code>으로 복사합니다(베이스 이미지에는 uv가 없습니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">WORKDIR /app</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이미지 안의 작업 폴더를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app</code>으로 정합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY pyproject.toml uv.lock ./</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">라이브러리 목록과 잠금 파일 두 개만 먼저 복사합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RUN uv sync --locked --no-dev --no-cache</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">잠금 파일의 버전 그대로(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--locked</code>), 테스트 도구 없이(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-dev</code>), 내려받은 파일을 남기지 않고(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-cache</code>) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app/.venv</code>에 설치합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY serving_app ./serving_app</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버 코드를 설치 뒤에 복사합니다. 코드만 고쳐 다시 빌드하면 위 두 단계는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CACHED</code>로 넘어갑니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ENV PATH=&quot;/app/.venv/bin:&#36;PATH&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv run</code> 없이 가상환경의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uvicorn</code>을 바로 실행하게 합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVICE_DB_PATH=/app/data/service.db</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">키 DB 파일의 위치입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">EXPOSE 8080</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너가 쓰는 포트를 적어 두는 표시입니다. 실제로 여는 것은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run -p</code>입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CMD [&quot;uvicorn&quot;, &quot;serving_app.managed_main:app&quot;, &quot;--host&quot;, &quot;0.0.0.0&quot;, &quot;--port&quot;, &quot;8080&quot;]</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서빙 API 앱을 8080 포트로 띄웁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.0.0.0</code>이라야 컨테이너 밖에서 들어온 요청을 받습니다.</td>
</tr>
</tbody>
</table>
</div>

### 2-2. 설정은 환경 변수로

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">장소마다 다른 값(vLLM 주소, 키)은 이미지에 적지 않고 환경 변수에서 읽습니다. 아래는 이 교안이 쓰는 설정만 보여 줍니다.</p>
</div>

```python
# core/config.py
class Settings(BaseSettings):
    """환경 변수에서 읽는 설정. 필드 이름을 대문자로 쓴 것이 환경 변수 이름이다(vllm_base_url -> VLLM_BASE_URL)."""

    # 서버를 실행한 폴더에서 위로 올라가며 찾은 .env(실습자료/.env)도 읽는다. .env 의 다른 키는 무시한다(extra="ignore").
    # 값이 틀려 서버가 멈출 때 오류 메시지에 입력값을 찍지 않는다(hide_input_in_errors). 키가 로그에 남지 않게 한다
    model_config = SettingsConfigDict(
        env_file=find_dotenv(usecwd=True) or None,
        extra="ignore",
        hide_input_in_errors=True,
    )

    # vLLM 서버 주소. 같은 파드에서는 127.0.0.1:8000, 배포 단원에서는 다른 파드의 주소로 바꾼다
    vllm_base_url: str = "http://127.0.0.1:8000/v1"

    # vLLM 서버를 --api-key 로 띄울 때 쓴 키. 기본값이 없으므로 반드시 정해야 한다(사용자에게는 알려 주지 않는다)
    vllm_api_key: str

    # vLLM 응답을 기다리는 최대 시간(초). 넘기면 시간 초과 오류가 난다
    vllm_timeout_seconds: float = 120

    # 서비스 키와 사용량을 저장할 SQLite 파일. 서버를 실행한 폴더 기준 상대 경로이고, 배포할 때는 환경 변수로 바꾼다
    service_db_path: Path = Path("output/service.db")

    # 관리자 키. 관리자 주소를 여는 앱만 쓰므로 기본값은 None 이고, 그 앱이 없으면 멈춘다
    admin_api_key: str | None = None

    # (이 교안이 쓰지 않는 설정은 생략)


# 여기서 값을 읽는다. 서버가 뜰 때 한 번만 읽으므로, 값을 바꾸려면 서버를 다시 띄운다. 모든 모듈이 이 settings 를 같이 쓴다
settings = Settings()
```

```python
# dependencies/security.py
ADMIN_API_KEY = settings.admin_api_key
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Settings</code>는 환경 변수를 읽고, 필드 이름을 대문자로 쓴 것이 환경 변수 이름입니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_base_url</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code>). 그래서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run -e</code>로 준 값이 그대로 서버 설정이 됩니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">환경 변수</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">없으면</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">3절에서 줄 값</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">교안 02에서 줄 값</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">기본값 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8000/v1</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:9/v1</code>(아무도 듣지 않는 포트)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">GPU 파드의 proxy 주소</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버가 뜰 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ValidationError</code>로 멈춤</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">local-test</code>(연습용 값)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">GPU 파드에 준 vLLM 키</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버가 뜰 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RuntimeError</code>로 멈춤</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">local-admin-key-for-practice</code>(연습용 값)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">노트북이 만든 관리자 키</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">기본값 120</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">주지 않음</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">90</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">빌드 때(Dockerfile의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">ENV</code>):</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVICE_DB_PATH</code>는 이미지에 적혀, 모든 컨테이너가 같은 값을 받습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 때(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker run -e</code>):</strong> 위 테이블의 값은 컨테이너를 띄울 때마다 줍니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">.dockerignore</code></strong>는 빌드 컨텍스트에서 뺄 파일과 폴더를 적는 파일이고, 저장소 맨 위에 둡니다. 여기 적은 파일은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY</code>로도 이미지에 들어가지 않습니다. 항목마다 빼는 이유는 파일의 주석에 적었습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving-deploy/.dockerignore</strong></div>

```text
# 이미지에 넣지 않을 것(빌드 컨텍스트에서 뺀다). 경로는 저장소 맨 위 기준이고, 아래 폴더까지 빼려면 **/ 를 붙인다

# git 기록 폴더(크기만 키운다)
.git
# 워크플로 파일(서빙 API 가 실행할 때 쓰지 않는다)
.github
workflows
# 내 PC 의 가상환경(이미지 안에서는 uv sync 로 따로 설치한다)
.venv
# 파이썬 캐시 파일(**/ 를 붙여 아래 폴더의 것까지 뺀다)
**/__pycache__
**/*.pyc
# 키를 적은 파일(이미지를 공개 패키지로 올리므로 이미지 안에 키가 없어야 한다)
.env
deploy/.env
# 실행 결과 폴더(크기만 키운다)
output
# 테스트와 설명 문서(서빙 API 가 실행할 때 쓰지 않는다)
tests
*.md
```

## 3. VM에서 빌드하고 실행하기

### 3-1. 이미지 빌드

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM에서 이미지를 빌드해 실제로 뜨는지 먼저 확인합니다. 태그 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">local</code>은 VM에서 직접 만든 이미지라는 표시입니다.</p>
<p style="margin:6px 0;">빌드 전에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">free -h</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Swap</code> 줄이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">2.0Gi</code>인지 봅니다(e2-micro는 메모리가 1GB). 0이면 부록 01(GCP VM 준비) 7절대로 스왑을 켭니다. 그래도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Killed</code>로 멈추면 4, 5절을 먼저 하고, 3-2 마지막 줄의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api:local</code>을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/&lt;소문자 아이디&gt;/serving-deploy:latest</code>로 바꿔 실행합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
free -h
docker build -t serving-api:local .
docker images serving-api
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">free -h</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">메모리(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Mem</code>)와 스왑(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Swap</code>)의 양을 읽기 쉬운 단위(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-h</code>)로 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker build -t serving-api:local .</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.</code>)를 빌드 컨텍스트로 이미지를 만들고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-t</code>로 이름 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api</code>와 태그 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">local</code>을 붙입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker images serving-api</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이름이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api</code>인 이미지 목록을 보여 줍니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">빌드가 오류 없이 끝나고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker images</code> 목록에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api:local</code>이 생기면 됩니다.</p>
</div>

### 3-2. 환경 변수를 주어 실행

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">2-2의 테이블대로 세 환경 변수를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-e</code>로 주어 실행합니다. 두 키 값은 <strong style="font-size:1.13em;color:#1f3a5f;">연습용 값</strong>이고 실제 배포 키가 아닙니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
docker run -d --name serving-api -p 8080:8080 \
  -e VLLM_BASE_URL=http://127.0.0.1:9/v1 \
  -e VLLM_API_KEY=local-test \
  -e ADMIN_API_KEY=local-admin-key-for-practice \
  serving-api:local
sleep 8   # e2-micro 에서 서버가 뜨기까지 3~5초 걸립니다
docker ps
docker logs serving-api
curl -sS localhost:8080/healthz
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run ... serving-api:local</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3-1에서 만든 이미지로 컨테이너를 실행합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-d</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">백그라운드에서 실행하고 터미널을 바로 돌려줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--name serving-api</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너 이름입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">logs</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">rm</code>에서 이 이름을 씁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-p 8080:8080</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><strong style="font-size:1.13em;color:#1f3a5f;">VM 포트:컨테이너 포트</strong>입니다. VM의 8080을 컨테이너의 8080으로 보냅니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-e 이름=값</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너 안에 환경 변수를 넣습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">줄 끝의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">\</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">명령이 다음 줄로 이어진다는 표시입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sleep 8</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버가 뜰 때까지 8초 기다립니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker logs serving-api</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너의 로그를 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">curl -sS 주소</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">HTTP 요청을 보내 응답 본문을 출력합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-s</code>는 진행 표시를 숨기고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-S</code>는 오류 메시지는 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">localhost:8080/healthz</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">VM 자신의 8080 포트에 있는 상태 확인 주소입니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker ps</code>의 STATUS가 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Up</code>:</strong> 서버가 떠 있습니다. 목록에 없으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker logs serving-api</code>의 마지막 줄을 봅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>를 빠뜨리면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_api_key</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field required</code>가, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code>를 빠뜨리면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RuntimeError: ADMIN_API_KEY 가 없습니다</code>가 보입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker logs</code>의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>:</strong> CMD의 주소로 떴습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">curl</code>의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">{&quot;status&quot;:&quot;ok&quot;}</code>:</strong> VM 안에서 서빙 API에 닿았습니다.</li>
</ul>
</div>

### 3-3. 내 PC에서 요청 보내기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 PC 노트북에서 VM의 상태 확인 주소를 부릅니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# VM 의 상태 확인 주소를 부릅니다(키가 필요 없고, vLLM 은 부르지 않습니다)
health_response = httpx.get(f"{VM_URL}/healthz", timeout=REQUEST_TIMEOUT)
print(f"상태 코드: {health_response.status_code} | 본문: {health_response.json()}")

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">관리자 주소로 서비스 키를 하나 발급합니다. 관리자 키는 3-2에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-e ADMIN_API_KEY</code>로 준 값과 같아야 합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 3-2 의 docker run 에 준 연습용 관리자 키입니다(이 교안의 로컬 시험에서만 씁니다)
LOCAL_ADMIN_HEADERS = {"X-Admin-Key": "local-admin-key-for-practice"}

# 관리자 주소로 free 요금제 서비스 키를 발급합니다
issue_response = httpx.post(f"{VM_URL}/v1/admin/keys", headers=LOCAL_ADMIN_HEADERS, json={"user": "이미지 확인", "plan": "free"}, timeout=REQUEST_TIMEOUT)
issued = issue_response.json()

# 키 원문은 헤더에만 담고, 화면에는 상태 코드, key_id, 사용자, 요금제만 보입니다
local_headers = {"X-API-Key": issued["api_key"]}
issued_view = {"상태 코드": issue_response.status_code, "key_id": issued["key_id"], "user": issued["user"], "plan": issued["plan"]}
display(issued_view)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">상태 확인의 200, 발급의 201:</strong> 컨테이너 안의 FastAPI와 키 DB(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app/data/service.db</code>)가 동작합니다. 이 키는 컨테이너 안에만 있어, 컨테이너를 새로 띄우면 401이 됩니다.</li>
</ul>
</div>

## 4. 이미지 이름과 태그

### 4-1. GHCR 이미지 이름

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GHCR(GitHub Container Registry):</strong> GitHub이 운영하는 이미지 저장소(레지스트리)입니다. GitHub 계정만으로 이미지를 올리고 받을 수 있습니다.</p>
<p style="margin:6px 0;">VM에서 만든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api:local</code>은 그 VM에만 있으므로, 다른 컴퓨터에서 쓰려면 GHCR에 올립니다. 이미지 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">레지스트리 주소/계정/저장소:태그</code> 모양이고, 이 단원은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/&lt;소문자 아이디&gt;/serving-deploy:&lt;태그&gt;</code>를 씁니다. 이미지 이름에는 대문자를 쓸 수 없어 GitHub 아이디를 소문자로 바꿉니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# GHCR 이미지 이름은 소문자만 됩니다. GitHub 아이디에 대문자가 있어도 소문자로 바꿉니다
IMAGE_NAME = "ghcr.io/" + GITHUB_OWNER.lower() + "/serving-deploy"
print(IMAGE_NAME)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">출력된 이름의 가운데 부분이 이후 명령의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;소문자 아이디&gt;</code> 자리에 들어갈 값입니다.</p>
</div>

### 4-2. 커밋 sha 태그와 latest

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">태그</strong>는 이미지 하나를 가리키는 이름표입니다. 같은 태그로 새 이미지를 올리면 태그가 새 이미지로 옮겨 가고, 옛 이미지는 그 태그로 부를 수 없습니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">커밋 sha 태그:</strong> 이미지를 만든 커밋의 sha(커밋마다 다른 40자리 16진수)를 태그로 씁니다. 커밋마다 값이 달라 겹치지 않으므로, 올린 이미지마다 자기 이름이 남습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">latest</code>:</strong> 올릴 때마다 가장 최근 이미지로 옮겨 가는 태그입니다.</li>
</ul>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/13_image_tags.png" width="1050" alt="커밋마다 sha 태그가 하나씩 남고 latest 태그만 가장 최근 이미지로 옮겨 가는 모습">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">커밋 세 개를 차례로 올리면 이렇게 됩니다(sha는 앞 6자리만 적은 예시).</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">commit 1을 올리면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">a1b2c3</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>가 같은 이미지를 가리킵니다.</li>
<li style="margin:3px 0;">commit 2를 올리면 새 이미지에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">d4e5f6</code>이 붙고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>는 이 이미지로 옮겨 갑니다. commit 1 이미지는 여전히 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">a1b2c3</code>으로 부를 수 있습니다.</li>
<li style="margin:3px 0;">commit 3에 문제가 생기면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">d4e5f6</code> 이미지를 실행해 commit 2 상태로 되돌립니다.</li>
</ol>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>만 붙이면 옛 이미지를 부를 이름이 없어 되돌릴 수 없습니다. 그래서 워크플로는 두 태그를 함께 붙이고, 교안 03의 배포는 sha 태그로 합니다.</p>
</div>

## 5. GitHub Actions로 이미지 올리기

### 5-1. image 워크플로 읽기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">main에 push하면 GitHub Actions가 이미지를 빌드해 GHCR에 올리게 합니다. 줄마다 하는 일은 파일의 주석에 적었습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving-deploy/workflows/image.yml</strong></div>

```yaml
# 이미지를 빌드해 GHCR 에 올리는 워크플로(교안 01). 이 파일을 저장소의 .github/workflows/ 로 복사하면 GitHub 이 실행한다(workflows/ 에 있는 동안은 실행되지 않는다)

name: image                                           # Actions 화면에 보일 워크플로 이름

# 이 워크플로를 실행할 때
on:
  # main 브랜치에 push 하면 실행한다
  push:
    branches: [main]
  workflow_dispatch:                                  # Actions 화면의 Run workflow 버튼으로도 실행할 수 있게 한다

# 이 실행이 쓰는 토큰(GITHUB_TOKEN)의 권한
permissions:
  contents: read                                      # 저장소 코드는 읽기만 한다
  packages: write                                     # GHCR 에 이미지를 올릴 수 있게 한다(없으면 올리는 단계에서 거절된다)

# 실행할 작업 목록(작업 하나: build)
jobs:
  build:
    runs-on: ubuntu-latest                            # GitHub 이 준비한 우분투 러너(실행마다 새로 받는 빈 컴퓨터)에서 실행한다
    timeout-minutes: 15                               # 15분이 넘으면 작업을 멈춘다
    # 위에서부터 차례로 실행할 단계
    steps:
      # 저장소 파일을 러너로 받아 온다
      - name: 저장소 받기
        uses: actions/checkout@v7

      # 이미지 이름을 정해 이 단계의 출력(name)으로 남긴다
      - name: 이미지 이름 정하기
        id: image                                     # 뒤 단계가 steps.image.outputs.name 으로 이 단계의 출력을 읽게 붙이는 이름
        run: echo "name=ghcr.io/${GITHUB_REPOSITORY,,}" >> "$GITHUB_OUTPUT"   # "계정/저장소"를 소문자로 바꿔(,,) name=값 한 줄을 출력 파일에 쓴다(이미지 이름은 소문자만 된다)

      # Docker 의 확장 빌드 도구 buildx 를 준비한다. 아래 type=gha 캐시는 buildx 로 빌드할 때만 쓸 수 있다
      - name: 빌드 도구(buildx) 준비
        uses: docker/setup-buildx-action@v4

      # docker login 과 같다. GHCR 에 로그인한다
      - name: GHCR 로그인
        uses: docker/login-action@v4
        with:
          registry: ghcr.io                           # 로그인할 레지스트리
          username: ${{ github.actor }}               # 이 실행을 시작한 GitHub 사용자
          password: ${{ secrets.GITHUB_TOKEN }}       # 실행마다 GitHub 가 주는 토큰(따로 등록하지 않음)

      # 이미지를 빌드하고 GHCR 에 올린다
      - name: 빌드하고 올리기
        uses: docker/build-push-action@v7
        with:
          context: .                                  # 저장소 폴더를 빌드 컨텍스트로 쓴다(그 폴더의 Dockerfile 로 빌드)
          push: true                                  # tags 의 이름을 모두 GHCR 에 올린다
          # 태그 둘: 커밋 sha(github.sha, 이 실행을 시작한 커밋. 되돌릴 때 부르는 이름)와 latest(가장 최근)
          tags: |
            ${{ steps.image.outputs.name }}:${{ github.sha }}
            ${{ steps.image.outputs.name }}:latest
          cache-from: type=gha                        # 지난 실행이 GitHub Actions 캐시(GHCR 이 아님)에 남긴 레이어를 가져온다
          cache-to: type=gha,mode=max                 # 이번 실행의 모든 레이어를 GitHub Actions 캐시에 남긴다(GHCR 에는 이미지만 올라간다)
```

### 5-2. 워크플로 켜고 push

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GitHub은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/</code> 안의 워크플로만 실행합니다. 내 PC 터미널에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/image.yml</code>을 그 폴더로 복사해 커밋하고 push합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
pushd ~/serving-deploy

cp workflows/image.yml .github/workflows/

git add .github/workflows/image.yml
git commit -m "이미지 빌드 워크플로 켜기"
git push
popd
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pushd 폴더</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 폴더를 기억해 두고 그 폴더로 이동합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cp 파일 폴더/</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">파일을 그 폴더로 복사합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git add 파일</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">그 파일의 변경을 다음 커밋에 담습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git commit -m &quot;메시지&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">담은 변경을 커밋으로 저장합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git push</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">새 커밋을 GitHub 저장소로 올립니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">popd</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pushd</code>가 기억해 둔 폴더로 돌아갑니다.</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">push가 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">refusing to allow ... without workflow scope</code>로 거절되면:</strong> GitHub CLI는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">gh auth refresh -s workflow</code>로, 토큰은 workflow 권한을 더한 뒤 다시 push합니다.</li>
</ul>
</div>

### 5-3. GitHub 화면에서 빌드 로그와 패키지 확인

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">push로 시작된 image 워크플로의 빌드 로그를 보고, 올라간 패키지의 공개 범위를 확인합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 저장소 위쪽 Actions 탭 &gt; 왼쪽 image &gt; 맨 위 실행 &gt; build</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/serving-deploy/actions</code></li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/14_actions_image_build.png" width="1050" alt="GitHub Actions의 image 실행 build 작업 화면. 빨간 사각형은 왼쪽 목록의 build 작업(초록 체크), 펼친 빌드하고 올리기 단계 제목, 그 단계 끝부분에서 ghcr.io로 시작하는 두 태그(커밋 sha, latest)를 올리는 줄">

<p style="margin:6px 0;">build 작업과 단계마다 초록 체크가 붙고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">빌드하고 올리기</code> 로그 끝에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/</code>로 시작하는 두 태그를 올리는 줄이 있으면 됩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">denied</code>로 멈췄으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">permissions: packages: write</code> 줄부터 확인합니다.</p>
</div>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">패키지 공개 범위:</strong> VM은 로그인하지 않고 이미지를 받으므로 패키지가 Public이어야 합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 저장소 첫 화면 오른쪽 Packages &gt; serving-deploy</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/users/&lt;GitHub 아이디&gt;/packages/container/package/serving-deploy</code></li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/15_ghcr_package_public.png" width="1050" alt="GHCR의 serving-deploy 패키지 화면. 빨간 사각형은 패키지 이름 옆의 Public 표시와, latest와 40자리 sha 태그가 있는 태그 목록">

<p style="margin:6px 0;">패키지 이름 옆에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Public</code>이 있고, 태그 목록에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>와 40자리 sha 태그가 있으면 됩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Private</code>이면 아래 순서로 바꿉니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 패키지 화면 오른쪽 Package settings &gt; 맨 아래 Danger Zone의 Change visibility &gt; Public 고르기 &gt; 패키지 이름 입력해 확인</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/users/&lt;GitHub 아이디&gt;/packages/container/serving-deploy/settings</code></li>
</ul>
</div>

## 6. 핵심 정리

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서빙 API Dockerfile:</strong> 베이스 이미지, uv 복사, 잠금 파일 복사와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv sync --locked --no-dev</code>, 서버 코드 복사, CMD(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.0.0.0:8080</code>) 순서입니다. 자주 바뀌는 서버 코드를 아래에 둡니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">잠금 파일:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv.lock</code>에 정확한 버전을 적어, 어디서 빌드해도 같은 라이브러리가 들어갑니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정은 실행 때:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run -e</code>로 넣고, 키 파일은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.dockerignore</code>로 뺍니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이미지 이름과 태그:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/&lt;소문자 아이디&gt;/serving-deploy:&lt;태그&gt;</code>. sha 태그는 되돌릴 이름이 되고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>는 가장 최근 이미지로 옮겨 갑니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">image 워크플로:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">packages: write</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_TOKEN</code>으로 GHCR에 로그인하고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type=gha</code> 캐시로 빌드해 두 태그로 올립니다. 패키지가 Public이어야 VM이 로그인 없이 받습니다.</li>
</ul>
<p style="margin:6px 0;">교안 02도 8080 포트를 쓰므로 이 교안의 컨테이너와 이미지를 지웁니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
docker rm -f serving-api
docker rmi serving-api:local
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker rm -f 이름</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너를 지웁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-f</code>는 실행 중이어도 멈춘 뒤 지웁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker rmi 이름:태그</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이미지를 지웁니다. 그 이미지의 컨테이너를 먼저 지워야 합니다.</td>
</tr>
</tbody>
</table>
</div>